# Concrete QC project - Step 1c (EDA: how does the concrete behave?)

Goal: describe the behaviour of the concrete - strength, variability, strength gain, slump, density, durability and trends.
We only DESCRIBE here. We do not try to explain why yet (that is Step 2).

We use the clean files from Step 1b. Every chart is also saved in `reports/figures`.

### Step 1: Load the clean files

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project = Path.cwd().parent   # the notebook runs from the notebooks folder, so the project is one level up
processed = project / "data" / "processed"
figures = project / "reports" / "figures"

cubes = pd.read_csv(processed / "cubes_clean.csv")
mixes = pd.read_csv(processed / "mix_designs.csv")
durability = pd.read_csv(processed / "durability_clean.csv")

cubes["date_cast"] = pd.to_datetime(cubes["date_cast"])
durability["date_cast"] = pd.to_datetime(durability["date_cast"])

print("Cubes:", cubes.shape)
print("Durability:", durability.shape)

### Step 2: Put the mixes in a sensible order

From the weakest to the strongest, with the CRCP (no fly ash) at the end. We use this order in every table and chart.

In [ ]:
mix_order = ["AFR-15/20", "AFR-20/20", "AFR-25/20", "AFR-30/20", "AFR-35/20", "AFR-40/20",
             "AFR-W30/20", "AFR-W40/20", "AFR-W50/20", "AFR-W50/14", "AFR-W50/20HS", "AFR-35/28CRCP"]

# shorter names for the charts (AFR-30/20 -> 30/20)
short_name = {mix: mix.replace("AFR-", "") for mix in mix_order}

day7 = cubes[cubes["age_group"] == "7 days"]
day28 = cubes[cubes["age_group"] == "28 days"]
print("7-day results:", len(day7), " 28-day results:", len(day28))

## PART A - STRENGTH

### Step 3: 28-day strength per mix

- mean = average strength
- std = standard deviation (how much the results spread)
- cov = std / mean x 100. Below about 10-12% is good control, above 20% is poor control

In [ ]:
strength_table = day28.groupby("mix_id")["strength"].agg(["count", "mean", "std", "min", "max"])
strength_table["cov_percent"] = strength_table["std"] / strength_table["mean"] * 100
strength_table = strength_table.reindex(mix_order).round(1)
print(strength_table)

### Step 4: How far above the class are we, and how many fail?

- `times_class` = average strength / class (1.3 means 30% above the class)
- `fail_percent` = % of 28-day results below the class

In [ ]:
class_table = day28.groupby("mix_id").agg(
    strength_class=("strength_class", "first"),
    mean_strength=("strength", "mean"),
    results=("strength", "count"),
    fails=("passed", lambda x: (x == False).sum()),
)
class_table["times_class"] = class_table["mean_strength"] / class_table["strength_class"]
class_table["fail_percent"] = class_table["fails"] / class_table["results"] * 100
class_table = class_table.reindex(mix_order).round(2)
print(class_table)

### Step 5: Chart - 28-day strength per mix

Each box shows the middle half of the results. The red line is the class the concrete must reach.

In [ ]:
data = [day28.loc[day28["mix_id"] == mix, "strength"] for mix in mix_order]
classes = [mixes.loc[mixes["mix_id"] == mix, "strength_class"].iloc[0] for mix in mix_order]

plt.figure(figsize=(12, 5))
plt.boxplot(data)
for i, c in enumerate(classes, start=1):
    plt.plot([i - 0.35, i + 0.35], [c, c], color="red")
plt.xticks(range(1, len(mix_order) + 1), [short_name[m] for m in mix_order], rotation=40)
plt.ylabel("28-day strength (MPa)")
plt.title("28-day cube strength per mix (red line = class)")
plt.tight_layout()
plt.savefig(figures / "01_strength_per_mix.png", dpi=150)
plt.show()

### Step 6: Strength gain from 7 to 28 days

We match the 7-day and 28-day result of the SAME pour (same LR, element and cast date),
then work out 7-day / 28-day. A ratio of 0.75 means the concrete had 75% of its 28-day strength at 7 days.

In [ ]:
key = ["lr", "element", "date_cast", "mix_id"]
seven = day7.groupby(key)["strength"].mean().rename("strength_7")
twenty_eight = day28.groupby(key)["strength"].mean().rename("strength_28")

pairs = pd.concat([seven, twenty_eight], axis=1).dropna().reset_index()
pairs["ratio_7_28"] = pairs["strength_7"] / pairs["strength_28"]
print("Pours with both a 7-day and 28-day result:", len(pairs))

ratio_table = pairs.groupby("mix_id")["ratio_7_28"].agg(["count", "median"]).reindex(mix_order).round(2)
print(ratio_table)

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar([short_name[m] for m in mix_order], ratio_table["median"])
plt.ylim(0.5, 1)
plt.ylabel("7-day / 28-day strength")
plt.title("Strength gain: share of 28-day strength reached at 7 days (CRCP has no fly ash)")
plt.xticks(rotation=40)
plt.tight_layout()
plt.savefig(figures / "02_strength_gain_7_28.png", dpi=150)
plt.show()

## PART B - SLUMP

### Step 7: Is the slump within the design range?

Each mix has a design slump range on its D2 form (for example 75-125 mm).

In [ ]:
slump_table = pd.crosstab(cubes["mix_id"], cubes["slump_check"], normalize="index") * 100
slump_table = slump_table.reindex(mix_order).round(0)
slump_table["median_slump"] = cubes.groupby("mix_id")["measured_slump"].median()
slump_table["design_range"] = mixes.set_index("mix_id")["design_slump_min"].astype(str) + "-" + mixes.set_index("mix_id")["design_slump_max"].astype(str)
print(slump_table)

In [ ]:
data = [cubes.loc[cubes["mix_id"] == mix, "measured_slump"].dropna() for mix in mix_order]

plt.figure(figsize=(12, 5))
plt.boxplot(data)
for i, mix in enumerate(mix_order, start=1):
    low = mixes.loc[mixes["mix_id"] == mix, "design_slump_min"].iloc[0]
    high = mixes.loc[mixes["mix_id"] == mix, "design_slump_max"].iloc[0]
    plt.plot([i - 0.35, i + 0.35], [low, low], color="green")
    plt.plot([i - 0.35, i + 0.35], [high, high], color="red")
plt.xticks(range(1, len(mix_order) + 1), [short_name[m] for m in mix_order], rotation=40)
plt.ylabel("Measured slump (mm)")
plt.title("Measured slump per mix (green = design minimum, red = design maximum)")
plt.tight_layout()
plt.savefig(figures / "03_slump_per_mix.png", dpi=150)
plt.show()

## PART C - DENSITY

### Step 8: Cube density per mix

In [ ]:
density_table = cubes.groupby("mix_id")["density"].agg(["count", "mean", "std", "min", "max"]).reindex(mix_order).round(0)
print(density_table)

### Step 9: Does a denser cube give a stronger cube? (28 days)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(day28["density"], day28["strength"], s=8, alpha=0.4)
plt.xlabel("Cube density (kg/m3)")
plt.ylabel("28-day strength (MPa)")
plt.title("Density vs strength - all 28-day results")
plt.tight_layout()
plt.savefig(figures / "04_density_vs_strength.png", dpi=150)
plt.show()

print("Correlation (all mixes):", round(day28["density"].corr(day28["strength"]), 2))

## PART D - TRENDS OVER TIME

### Step 10: Has the 28-day strength changed over the contract?

We use the four mixes with the most results and group them per half year (H1 = Jan-Jun, H2 = Jul-Dec).

In [ ]:
day28_dated = day28.dropna(subset=["date_cast"]).copy()
day28_dated["half_year"] = day28_dated["date_cast"].dt.year.astype(str) + "-H" + ((day28_dated["date_cast"].dt.month > 6) + 1).astype(str)

main_mixes = ["AFR-20/20", "AFR-30/20", "AFR-40/20", "AFR-W40/20"]
trend = day28_dated[day28_dated["mix_id"].isin(main_mixes)]
trend_table = trend.pivot_table(index="half_year", columns="mix_id", values="strength", aggfunc="mean").round(1)
trend_count = trend.pivot_table(index="half_year", columns="mix_id", values="strength", aggfunc="count")
print("Average 28-day strength:")
print(trend_table)
print()
print("Number of results:")
print(trend_count)

In [ ]:
plt.figure(figsize=(11, 5))
for mix in main_mixes:
    plt.plot(trend_table.index, trend_table[mix], marker="o", label=short_name[mix])
plt.ylabel("Average 28-day strength (MPa)")
plt.title("28-day strength over time (half-year averages)")
plt.legend()
plt.xticks(rotation=40)
plt.tight_layout()
plt.savefig(figures / "05_strength_trend.png", dpi=150)
plt.show()

# Careful: a half-year with only a few results can jump up or down. Check the counts table above.

## PART E - DURABILITY

### Step 11: Durability overall

Spec (Table B6404/4): OPI above 9.3 = full acceptance, 9.0-9.3 = conditional, below 9.0 = reject. Sorptivity below 10.

In [ ]:
print(durability[["opi_mean", "sorptivity_mean", "porosity"]].describe().round(2))
print()
print("OPI judgement:")
print(durability["opi_judgement"].value_counts())
print()
print("Sorptivity judgement:")
print(durability["sorp_judgement"].value_counts())

### Step 12: Durability per structure and per element

In [ ]:
by_structure = durability.groupby("structure")[["opi_mean", "sorptivity_mean"]].agg(["count", "mean", "min", "max"]).round(2)
print(by_structure)
print()
by_element = durability.groupby("element_type")[["opi_mean", "sorptivity_mean"]].agg(["count", "mean"]).round(2)
print(by_element)

### Step 13: Durability over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].scatter(durability["date_cast"], durability["opi_mean"], s=15)
axes[0].axhline(9.3, color="green", linestyle="--", label="9.3 full acceptance")
axes[0].axhline(9.0, color="red", linestyle="--", label="9.0 reject below")
axes[0].set_ylabel("OPI")
axes[0].set_title("Oxygen permeability index (higher is better)")
axes[0].legend()

axes[1].scatter(durability["date_cast"], durability["sorptivity_mean"], s=15, color="orange")
axes[1].axhline(10, color="red", linestyle="--", label="10 maximum")
axes[1].set_ylabel("Sorptivity (mm/h^0.5)")
axes[1].set_title("Water sorptivity (lower is better)")
axes[1].legend()

plt.tight_layout()
plt.savefig(figures / "06_durability_over_time.png", dpi=150)
plt.show()

print(durability.groupby("year")[["opi_mean", "sorptivity_mean"]].agg(["count", "mean"]).round(2))

## PART F - FLY ASH vs NO FLY ASH

### Step 14: A first look

11 mixes have about 30% fly ash, only the CRCP has none. So this is a first look, not proof -
the CRCP is also a different element (road pavement) and a different mix.

In [ ]:
pairs["fly_ash"] = pairs["mix_id"].apply(lambda m: "no fly ash (CRCP)" if "CRCP" in m else "30% fly ash")
print(pairs.groupby("fly_ash")["ratio_7_28"].agg(["count", "median"]).round(2))

### Step 15: Write down what you found

Answer these in your own words before Step 2:
1. Which mixes are well controlled and which are not (look at cov_percent and fail_percent)?
2. How far above their class do the mixes run on average?
3. How much strength does the concrete have at 7 days compared with 28 days, and is the CRCP different?
4. Does the slump stay inside the design range? Which mixes are mostly too wet?
5. Is there a link between density and strength?
6. Has the strength changed over time for any mix?
7. How does the durability look, and which structures or elements are the best and the weakest?

### Answers - Step 1c

1. **Which mixes are well controlled and which are not?**
- Well controlled (cov about 9-13%, few fails): W50/20 (8.9%, 1.8% fail), W50/14 (10.4%), CRCP 35/28 (10.9%, 0% fail), W40/20 (11.1%, 0.5% fail), 40/20 (13.2%, 3.5% fail).
- Fair: 30/20 (16.5%, 4.2% fail), W30/20 (18.4%), W50/20 high slump (16.2%, 11.8% fail but only 17 results).
- Poor: 15/20 (46%, 15% fail), 20/20 (40.5%, 23% fail), 35/20 (31%, 29% fail but only 24 results), 25/20 (23%, 7.6% fail).
The low-strength mixes are the problem - their results swing from about 14 to 68 MPa.

2. **How far above their class do the mixes run on average?**
Every mix is above its class on average - from 1.16 x (35/20) to 1.84 x (15/20). Most run 25-45% above: 30/20 1.37 x, 40/20 1.34 x, W40/20 1.27 x. The CRCP runs 1.56 x above its 35 MPa class (54.7 MPa).

3. **How much strength at 7 days compared with 28 days, and is the CRCP different?**
The fly ash mixes have about 72-81% of their 28-day strength at 7 days (median 0.75). The CRCP (no fly ash) has 86%, so it gains strength faster early on - what we expect without fly ash.

4. **Does the slump stay inside the design range? Which mixes are mostly too wet?**
The pump mixes mostly stay inside (W40/20 97%, W30/20 90%, W50/14 88%, W50/20 HS 83%, W50/20 80%). The standard 75-125 mm mixes are often too wet: 40/20 is above the range 68% of the time (median 135 mm), CRCP 67% (140 mm), 30/20 50% (130 mm), 15/20 46%, 35/20 44%, 25/20 40%. Almost nothing is too dry, except a few 0 mm outliers.

5. **Is there a link between density and strength?**
Yes, but only moderate - correlation 0.40 for all 28-day results, so denser cubes tend to be stronger. It is strongest inside W50/20 HS (0.94), W50/14 (0.80) and CRCP (0.64), and weak for 20/20 (0.15), W50/20 (0.23) and W40/20 (0.25). Densities are very similar between mixes (about 2,360-2,405 kg/m3), except the CRCP at about 2,440.

6. **Has the strength changed over time for any mix?**
Yes. 30/20 dropped from about 43-45 MPa (2023-2024) to about 39 MPa (2025) and 34 MPa (second half of 2026, 28 results). 40/20 peaked at 63 MPa in early 2024 and is now about 48-50 MPa. W40/20 stayed steady at 50-52 MPa. 20/20 jumps around (21-44 MPa), but the early periods have very few results.

7. **How does the durability look, and which structures or elements are the best and the weakest?**
All 198 tests pass. OPI averages 10.32 (lowest 9.45, limit 9.3) and sorptivity 6.67 (highest 8.85, limit 10). Best: the Sky Circle deck (OPI 10.86, sorptivity 5.72) and decks in general (OPI 10.56). Weakest: parapets (OPI 10.02, lowest single results 9.45), pile caps (10.05) and the WB retaining wall (OPI 10.05, highest sorptivity 7.31). Durability has improved over time - OPI from 10.2 (2023) to 10.9 (2026) and sorptivity from 7.0 to 5.8.